# 05. 장애인콜택시 바로콜 예상 대기시간 모델링

목적: 네비게이터에서 장애인콜택시를 선택했을 때 표시할 **예상 대기시간**을 예측한다.

1차 모델 범위:

- 임차택시 바로콜 승차완료건
- 특장차 바로콜 승차완료건

1차 모델에서 제외:

- 취소건: `접수→승차` target 생성 불가
- 특장차 전일접수: 예정시각 기준 지연 문제라 바로콜과 target 정의가 다름
- 심야시간 사전예약: 표본 부족으로 참고 통계/주의 플래그 처리

예측 대상:

```text
target = 접수→승차_분
```

1차 입력 변수 원칙:

- `출발구`는 배차 가능한 차량 접근성과 직접 관련되므로 기본 피처로 사용
- `목적구`는 이동 방향, 회차 부담, 장거리/외곽 이동 가능성을 반영할 수 있으므로 1차 모델 피처로 포함
- `출발동`, `목적동`은 원천 데이터에는 보존하되, 고유값이 많아 과적합 위험이 있으므로 1차 모델에서는 제외하고 성능 비교 단계에서 추가 검토


극단 대기시간 처리 원칙:

- 180분 초과 대기는 전체 바로콜 접수→승차 대기시간의 약 p99.5 구간이므로 기본 학습 데이터에서 바로 제거하지 않음
- `is_extreme_wait = 접수_승차_분 > 180` 컬럼을 만들어 장시간 극단 사례를 별도로 표시
- 기본 모델은 전체 데이터를 사용하고, 비교 실험이 필요할 때만 `target_cap=180.0` 또는 클리핑 버전을 추가로 실행


계절 변수 처리 원칙:

- 데이터 기간은 365일이므로 월별/계절성 영향은 고려할 수 있음
- 다만 `season`은 `month`를 묶은 중복 파생 변수이므로 1차 모델 입력에서는 제외
- 1차 모델은 `month`만 사용하고, `season`은 해석용/비교 실험용으로만 보관


## 1. 라이브러리 및 경로 설정

In [3]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "calltaxi-DA":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "waiting_time"
REPORT_DIR = PROJECT_ROOT / "reports" / "waiting_time_model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RENTAL_PATH = DATA_DIR / "임차택시_대기시간_전처리.csv"
SPECIAL_PATH = DATA_DIR / "특장차_대기시간_전처리_접수유형분류.csv"

RANDOM_STATE = 42

RENTAL_PATH, SPECIAL_PATH

(PosixPath('/Users/blaumonde/calltaxi-DA/data/processed/임차택시_대기시간_전처리.csv'),
 PosixPath('/Users/blaumonde/calltaxi-DA/data/processed/특장차_대기시간_전처리_접수유형분류.csv'))

## 2. 모델링 데이터셋 생성

`차량유형`과 `호출유형`을 따로 두지 않고, 실제 데이터에 존재하는 조합을 하나의 컬럼으로 만든다.

```text
model_group = 임차택시_바로콜
model_group = 특장차_바로콜
```

```text
전체 데이터
→ 취소건 제외
→ 전일접수 제외
→ 심야시간 사전예약 제외
→ 미분류/분류 애매한 건 제외
→ 승차일시 있고 접수→승차 대기시간 계산 가능한 건만 사용
→ 임차택시 바로콜 + 특장차 바로콜
= 1,404,813건

| 구분 | 건수 | p90 | p95 | p99 | p99.5 | 최대 |
|---|---:|---:|---:|---:|---:|---:|
| 전체 | 1,404,813 | 94.0분 | 138.1분 | 168.6분 | 179.9분 | 249.9분 |
| 임차택시 바로콜 | 305,729 | 87.3분 | 145.0분 | 174.6분 | 189.9분 | 249.9분 |
| 특장차 바로콜 | 1,099,084 | 95.1분 | 133.9분 | 166.6분 | 177.6분 | 200.0분 |

In [4]:
def load_modeling_dataset(target_cap=None, extreme_wait_threshold=180.0):
    rental_cols = [
        "접수일시", "승차일시", "출발구", "출발동", "목적구", "목적동",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "임차택시_바로콜여부", "대기시간분석_포함여부",
    ]
    special_cols = [
        "접수일시", "승차일시", "출발구", "출발동", "목적구", "목적동",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "특장차_바로콜_후보여부", "특장차_접수유형_후보_최종",
    ]

    rental = pd.read_csv(RENTAL_PATH, usecols=rental_cols, low_memory=False)
    rental = rental[
        rental["임차택시_바로콜여부"].fillna(False).astype(bool)
        & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
    ].copy()
    rental["model_group"] = "임차택시_바로콜"

    special = pd.read_csv(SPECIAL_PATH, usecols=special_cols, low_memory=False)
    special = special[
        special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
        | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
    ].copy()
    special["model_group"] = "특장차_바로콜"

    keep_cols = [
        "model_group", "접수일시", "승차일시",
        "출발구", "출발동", "목적구", "목적동",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
    ]
    data = pd.concat([rental[keep_cols], special[keep_cols]], ignore_index=True)

    data["접수일시"] = pd.to_datetime(data["접수일시"], errors="coerce")
    data["승차일시"] = pd.to_datetime(data["승차일시"], errors="coerce")
    for col in ["접수_배차_분", "배차_승차_분", "접수_승차_분"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data[
        data["접수일시"].notna()
        & data["승차일시"].notna()
        & data["접수_승차_분"].notna()
        & data["접수_승차_분"].ge(0)
    ].copy()

    # 180분 초과 대기는 기본적으로 제거하지 않고 별도 표시한다.
    # 네비게이터에서는 장시간 대기 위험 자체가 중요한 정보이기 때문이다.
    data["is_extreme_wait"] = data["접수_승차_분"].gt(extreme_wait_threshold).astype("int8")

    # 비교 실험이 필요할 때만 target_cap을 적용한다.
    if target_cap is not None:
        data = data[data["접수_승차_분"].le(target_cap)].copy()

    data["target_min"] = data["접수_승차_분"]

    # 시간 파생 변수
    data["hour"] = data["접수일시"].dt.hour.astype("int16")
    data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
    data["month"] = data["접수일시"].dt.month.astype("int16")
    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype("int8")
    data["is_night"] = data["hour"].between(0, 6, inclusive="both").astype("int8")
    data["is_commute"] = (
        data["hour"].between(7, 9, inclusive="both")
        | data["hour"].between(17, 19, inclusive="both")
    ).astype("int8")
    data["season"] = data["month"].map({
        12: "겨울", 1: "겨울", 2: "겨울",
        3: "봄", 4: "봄", 5: "봄",
        6: "여름", 7: "여름", 8: "여름",
        9: "가을", 10: "가을", 11: "가을",
    })

    for col in ["출발구", "출발동", "목적구", "목적동", "season"]:
        data[col] = data[col].fillna("미상").astype(str)

    return data

data = load_modeling_dataset(target_cap=None)
data.shape

(1404813, 19)

In [5]:
# 극단 대기시간 규모 확인: 제거하지 않고 별도 표시만 한다.
extreme_wait_summary = (
    data.groupby("model_group")
    .agg(
        전체건수=("target_min", "count"),
        극단대기건수=("is_extreme_wait", "sum"),
        중앙값=("target_min", "median"),
        평균=("target_min", "mean"),
        최대값=("target_min", "max"),
    )
    .reset_index()
)
extreme_wait_summary["극단대기비율(%)"] = (
    extreme_wait_summary["극단대기건수"] / extreme_wait_summary["전체건수"] * 100
)

display(extreme_wait_summary.round(2))

,model_group,전체건수,극단대기건수,중앙값,평균,최대값,극단대기비율(%)
0,임차택시_바로콜,305729,2387,28.51,42.45,249.90,0.78
1,특장차_바로콜,1099084,4614,33.42,46.27,199.99,0.42


## 3. 데이터 기본 확인

## 모델링 데이터 컬럼 설명

| 컬럼명 | 의미 | 모델 사용 여부 |
|---|---|---|
| `model_group` | 모델에서 구분하는 차량·호출 유형. 현재는 `임차택시_바로콜`, `특장차_바로콜`로 구분 | 사용 |
| `접수일시` | 이용자가 콜을 접수한 시각 | 원천/파생용 |
| `승차일시` | 이용자가 실제 차량에 승차한 시각 | target 생성용 |
| `출발구` | 이용자의 출발지 자치구 | 사용 |
| `출발동` | 이용자의 출발지 행정동 | 보관, 1차 모델 제외 |
| `목적구` | 이용자의 목적지 자치구 | 사용 |
| `목적동` | 이용자의 목적지 행정동 | 보관, 1차 모델 제외 |
| `접수_배차_분` | 접수 후 차량이 배차되기까지 걸린 시간(분) | 해석용 |
| `배차_승차_분` | 배차 후 실제 승차까지 걸린 시간(분) | 해석용 |
| `접수_승차_분` | 접수 후 실제 승차까지 걸린 총 대기시간(분) | target 원천 |
| `is_extreme_wait` | `접수_승차_분 > 180` 여부. 상위 약 0.5% 수준의 극단 대기 표시 | 해석/검증용 |
| `target_min` | 최종 예측 대상. `접수_승차_분`과 동일 | 예측 대상 |
| `hour` | 접수일시에서 추출한 시간대. 0~23시 | 사용 |
| `dayofweek` | 접수일시에서 추출한 요일. 월=0, 화=1, ..., 일=6 | 사용 |
| `month` | 접수일시에서 추출한 월. 1~12월 | 사용 |
| `is_weekend` | 주말 여부. 토/일이면 1, 평일이면 0 | 사용 |
| `is_night` | 심야 여부. 00~06시 접수면 1, 아니면 0 | 사용 |
| `is_commute` | 출퇴근 시간 여부. 07~09시 또는 17~19시면 1, 아니면 0 | 사용 |
| `season` | 접수월을 기준으로 만든 계절 구분. 봄/여름/가을/겨울 | 보관, 1차 모델 제외 |

In [6]:
display(data.head())

summary = data.groupby("model_group")["target_min"].agg(
    건수="count",
    평균="mean",
    중앙값="median",
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
).round(2)

display(summary)

,model_group,접수일시,승차일시,출발구,출발동,목적구,목적동,접수_배차_분,배차_승차_분,접수_승차_분,is_extreme_wait,target_min,hour,dayofweek,month,is_weekend,is_night,is_commute,season
0,임차택시_바로콜,2025-01-01 05:56:10.000,2025-01-01 08:17:00,강동구,상일동,강동구,명일제2동,118.833333,22.0,140.833333,0,140.833333,5,2,1,0,1,0,겨울
1,임차택시_바로콜,2025-01-01 03:55:00.983,2025-01-01 07:15:00,성북구,삼선동,성북구,길음제2동,189.983617,10.0,199.983617,1,199.983617,3,2,1,0,1,0,겨울
2,임차택시_바로콜,2025-01-01 06:27:42.000,2025-01-01 08:30:00,서초구,방배3동,노원구,상계6.7동,117.300000,5.0,122.300000,0,122.300000,6,2,1,0,1,0,겨울
3,임차택시_바로콜,2025-01-01 07:15:29.733,2025-01-01 08:45:00,강동구,암사제3동,강동구,길동,81.504450,8.0,89.504450,0,89.504450,7,2,1,0,0,1,겨울
4,임차택시_바로콜,2025-01-01 08:45:22.857,2025-01-01 09:45:00,강동구,강일동,하남시,신장1동,39.619050,20.0,59.619050,0,59.619050,8,2,1,0,0,1,겨울


,건수,평균,중앙값,p75,p90,p95
model_group,,,,,,
임차택시_바로콜,305729,42.45,28.51,50.28,87.34,145.05
특장차_바로콜,1099084,46.27,33.42,57.33,95.13,133.94


## 4. Train / Validation / Test 분리

365일 데이터이므로 시간순 70/15/15 분리는 월별/계절성의 영향을 받을 수 있다.

따라서 1차 모델에서는 **월 × model_group 기준 층화 랜덤 분리**를 사용한다.

In [7]:
def stratified_split(data):
    strata = data["month"].astype(str) + "_" + data["model_group"].astype(str)

    train, temp = train_test_split(
        data,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=strata,
    )

    temp_strata = temp["month"].astype(str) + "_" + temp["model_group"].astype(str)
    valid, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_strata,
    )
    return train, valid, test

train, valid, test = stratified_split(data)

print(train.shape, valid.shape, test.shape)

split_summary = pd.DataFrame({
    "train": train["model_group"].value_counts(normalize=True),
    "valid": valid["model_group"].value_counts(normalize=True),
    "test": test["model_group"].value_counts(normalize=True),
}).round(4)

display(split_summary)

(983369, 19) (210722, 19) (210722, 19)


,train,valid,test
model_group,,,
특장차_바로콜,0.7824,0.7824,0.7824
임차택시_바로콜,0.2176,0.2176,0.2176


## 5. 평가 함수 정의

### 평가지표

- `MAE`: 예측 대기시간이 실제 대기시간과 평균적으로 몇 분 차이 나는지 보여준다.

- `Median_AE`: 일반적인 케이스에서 예측 오차가 어느 정도인지 보여준다.

- `RMSE`: 큰 예측 오차에 민감해서 장시간 대기를 크게 틀리는 경우를 확인하는 데 유용하다.

- `R2`: 모델이 대기시간의 변동을 어느 정도 설명하는지 보여준다.

- `Accuracy`: 전체 건수 중 장시간 대기 여부를 맞춘 비율이다.

- `Precision`: 모델이 장시간 대기라고 예측한 건 중 실제 장시간 대기였던 비율이다.

- `Recall`: 실제 장시간 대기 건 중 모델이 장시간 대기로 잡아낸 비율이다.

- `F1`: Precision과 Recall의 균형을 보여주는 지표다.

- `Confusion_Matrix`: 정상 대기와 장시간 대기를 각각 어떻게 맞추고 틀렸는지 보여준다.

In [ ]:
# 실제 대기시간과 예측 대기시간의 차이를 기준으로 회귀 모델 성능을 평가
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

# 기준 시간 이상을 장시간 대기로 보고, 장시간 대기 위험을 얼마나 잘 맞췄는지 평가
def risk_metrics(y_true, y_pred, threshold):
    actual = y_true >= threshold
    pred = y_pred >= threshold
    return {
        "threshold": threshold,
        "Accuracy": accuracy_score(actual, pred),
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0),
        "Confusion_Matrix": confusion_matrix(actual, pred, labels=[False, True]).tolist(),
    }

# 전체 데이터에 대해 회귀 성능과 장시간 대기 위험 예측 성능을 함께 평가
def evaluate_predictions(frame, pred, threshold):
    y = frame["target_min"].to_numpy()
    return {
        **regression_metrics(y, pred),
        **{f"risk_{k}": v for k, v in risk_metrics(y, pred, threshold).items()},
    }

# 임차택시 바로콜과 특장차 바로콜을 나누어 그룹별 모델 성능을 비교
def evaluate_by_group(frame, pred, threshold_by_group):
    tmp = frame[["model_group", "target_min"]].copy()
    tmp["prediction"] = pred
    rows = []
    for group, group_df in tmp.groupby("model_group"):
        y = group_df["target_min"].to_numpy()
        p = group_df["prediction"].to_numpy()
        threshold = threshold_by_group[group]
        rows.append({
            "model_group": group,
            "rows": len(group_df),
            **regression_metrics(y, p),
            **{f"risk_{k}": v for k, v in risk_metrics(y, p, threshold).items() if k != "Confusion_Matrix"},
        })
    return pd.DataFrame(rows)

## 6. Baseline 모델

Baseline 1은 전체 중앙값, Baseline 2는 `model_group × 접수시간대 × 출발구 × 목적구` 중앙값 기반 백오프 모델이다.

```text
1순위: model_group × hour × 출발구 × 목적구
2순위: model_group × hour × 출발구
3순위: model_group × hour
4순위: model_group 전체 중앙값

```text
Baseline 2는 조건별 중앙값을 사용하는 규칙 기반 모델이다.
희귀 조합의 불안정한 중앙값을 피하기 위해 min_count를 적용했고,
50, 100, 200 중 validation MAE가 가장 낮은 값을 최종 baseline 기준으로 선택한다.

In [11]:
# Baseline 1: 전체 중앙값 모델
# 모든 데이터에 train set의 전체 접수→승차 중앙값을 동일하게 예측한다.
overall_median = train["target_min"].median()

valid_pred_baseline_1 = np.full(len(valid), overall_median)
test_pred_baseline_1 = np.full(len(test), overall_median)


# Baseline 2: 조건별 중앙값 백오프 모델
# RandomForest 1차 모델의 주요 입력 변수 중
# model_group, hour, 출발구, 목적구를 사용해 조건별 과거 중앙값을 예측값으로 사용한다.
def baseline_group_hour_origin_dest(train, target_frame, min_count=100):
    # 가장 넓은 백오프: model_group별 중앙값
    global_medians = (
        train.groupby("model_group")["target_min"]
        .median()
        .to_dict()
    )

    # 3순위 백오프: model_group × hour 중앙값
    group_hour_medians = (
        train.groupby(["model_group", "hour"])["target_min"]
        .median()
        .to_dict()
    )

    # 2순위 백오프: model_group × hour × 출발구 중앙값
    origin_medians = (
        train.groupby(["model_group", "hour", "출발구"])
        .agg(median=("target_min", "median"), count=("target_min", "size"))
        .reset_index()
    )
    origin_medians = origin_medians[origin_medians["count"].ge(min_count)]

    origin_map = {
        (r["model_group"], int(r["hour"]), r["출발구"]): float(r["median"])
        for _, r in origin_medians.iterrows()
    }

    # 1순위: model_group × hour × 출발구 × 목적구 중앙값
    detailed = (
        train.groupby(["model_group", "hour", "출발구", "목적구"])
        .agg(median=("target_min", "median"), count=("target_min", "size"))
        .reset_index()
    )
    detailed = detailed[detailed["count"].ge(min_count)]

    detailed_map = {
        (r["model_group"], int(r["hour"]), r["출발구"], r["목적구"]): float(r["median"])
        for _, r in detailed.iterrows()
    }

    preds = []

    for r in target_frame[["model_group", "hour", "출발구", "목적구"]].itertuples(index=False):
        detailed_key = (r.model_group, int(r.hour), r.출발구, r.목적구)
        origin_key = (r.model_group, int(r.hour), r.출발구)
        group_hour_key = (r.model_group, int(r.hour))

        if detailed_key in detailed_map:
            preds.append(detailed_map[detailed_key])
        elif origin_key in origin_map:
            preds.append(origin_map[origin_key])
        elif group_hour_key in group_hour_medians:
            preds.append(group_hour_medians[group_hour_key])
        else:
            preds.append(global_medians[r.model_group])

    return np.array(preds, dtype=float)


# 장시간 대기 기준
# 전체 기준: train set의 접수→승차 90분위수
long_wait_threshold = train["target_min"].quantile(0.90)

# 차량·호출유형별 기준: 임차택시 바로콜, 특장차 바로콜 각각의 90분위수
threshold_by_group = (
    train.groupby("model_group")["target_min"]
    .quantile(0.90)
    .to_dict()
)


# Baseline 성능 비교
baseline_rows = []

# Baseline 1
baseline_rows.extend([
    {
        "model": "Baseline 1 전체 중앙값",
        "split": "valid",
        "min_count": None,
        **regression_metrics(valid["target_min"], valid_pred_baseline_1),
    },
    {
        "model": "Baseline 1 전체 중앙값",
        "split": "test",
        "min_count": None,
        **regression_metrics(test["target_min"], test_pred_baseline_1),
    },
])

# Baseline 2: min_count 후보 비교
for min_count in [50, 100, 200]:
    valid_pred_baseline_2 = baseline_group_hour_origin_dest(
        train,
        valid,
        min_count=min_count,
    )
    test_pred_baseline_2 = baseline_group_hour_origin_dest(
        train,
        test,
        min_count=min_count,
    )

    baseline_rows.extend([
        {
            "model": "Baseline 2 조건별 중앙값 백오프",
            "split": "valid",
            "min_count": min_count,
            **regression_metrics(valid["target_min"], valid_pred_baseline_2),
        },
        {
            "model": "Baseline 2 조건별 중앙값 백오프",
            "split": "test",
            "min_count": min_count,
            **regression_metrics(test["target_min"], test_pred_baseline_2),
        },
    ])

baseline_results = pd.DataFrame(baseline_rows)

display(baseline_results.round(4))

,model,split,min_count,MAE,Median_AE,RMSE,R2
0,Baseline 1 전체 중앙값,valid,NaN,23.9640,13.4304,38.4253,-0.1307
1,Baseline 1 전체 중앙값,test,NaN,23.8425,13.4113,38.2438,-0.1286
2,Baseline 2 조건별 중앙값 백오프,valid,50.0,17.5387,9.8091,29.2511,0.3448
3,Baseline 2 조건별 중앙값 백오프,test,50.0,17.5092,9.8029,29.2020,0.3420
4,Baseline 2 조건별 중앙값 백오프,valid,100.0,17.8447,9.9102,29.7849,0.3207
5,Baseline 2 조건별 중앙값 백오프,test,100.0,17.7770,9.9074,29.6397,0.3221
6,Baseline 2 조건별 중앙값 백오프,valid,200.0,18.0796,10.0199,30.1475,0.3040
7,Baseline 2 조건별 중앙값 백오프,test,200.0,18.0302,10.0063,30.0263,0.3043


## 7. RandomForestRegressor 1차 모델

Baseline 2는 차량·호출유형, 접수시간대, 출발구, 목적구별 과거 중앙값을 사용하는 규칙 기반 모델이다.  
이제 1차 머신러닝 모델로 `RandomForestRegressor`를 학습한다.

RandomForestRegressor는 여러 개의 의사결정나무를 만들고, 각 나무의 예측값을 평균내어 최종 대기시간을 예측하는 앙상블 모델이다.  
시간대, 요일, 월, 차량·호출유형, 출발구, 목적구처럼 여러 조건이 함께 작용하는 비선형 패턴을 학습할 수 있다.

1차 모델에서는 다음 변수를 사용한다.

```text
model_group
출발구
목적구
hour
dayofweek
month
is_weekend
is_night
is_commute

In [13]:
from sklearn.ensemble import RandomForestRegressor

FEATURES = [
    "model_group", "출발구", "목적구",
    "hour", "dayofweek", "month", "is_weekend", "is_night", "is_commute",
]

# 목적구는 1차 모델에 포함한다.
# 목적동은 원천 데이터에는 보존하지만, 고유값이 많아 과적합 위험이 있으므로 1차 모델에서는 제외한다.
# season은 month에서 파생된 중복 변수이므로 1차 모델에서는 제외하고 해석/비교용으로만 보관한다.

def build_rf_pipeline(params=None):
    categorical_cols = ["model_group", "출발구", "목적구"]
    numeric_cols = ["hour", "dayofweek", "month", "is_weekend", "is_night", "is_commute"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1,
            ), categorical_cols),
            ("num", "passthrough", numeric_cols),
        ],
        verbose_feature_names_out=False,
    )

    default_params = {
        "n_estimators": 100,
        "max_depth": 18,
        "min_samples_leaf": 20,
        "max_features": 1.0,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "verbose": 1,
    }

    if params:
        default_params.update(params)

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(**default_params)),
    ])


rf_model = build_rf_pipeline()

rf_model.fit(train[FEATURES], train["target_min"])

valid_pred_rf = rf_model.predict(valid[FEATURES])
test_pred_rf = rf_model.predict(test[FEATURES])

rf_results = pd.DataFrame([
    {
        "model": "RandomForest 기본",
        "split": "valid",
        **regression_metrics(valid["target_min"], valid_pred_rf),
    },
    {
        "model": "RandomForest 기본",
        "split": "test",
        **regression_metrics(test["target_min"], test_pred_rf),
    },
])

display(rf_results.round(4))

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.9s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   35.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished


,model,split,MAE,Median_AE,RMSE,R2
0,RandomForest 기본,valid,16.5870,10.8694,25.6213,0.4973
1,RandomForest 기본,test,16.5872,10.8687,25.6320,0.4930


## 8. 하이퍼파라미터 튜닝

큰 GridSearch나 베이지안 최적화 대신, `ParameterSampler` 기반의 소규모 RandomizedSearch를 사용한다.

데이터가 약 140만 건으로 크기 때문에 가능한 모든 조합을 탐색하는 방식은 학습 시간이 오래 걸릴 수 있다. 따라서 일부 하이퍼파라미터 조합만 랜덤하게 선택해 validation set에서 성능을 비교한다.

1차 선택 기준은 `validation MAE`로 둔다.  
MAE는 예측 대기시간이 실제 대기시간과 평균적으로 몇 분 차이 나는지 보여주는 가장 직관적인 지표이기 때문이다.

다만 최종 모델 선택에서는 MAE만 보지 않고, 아래 지표를 함께 확인한다.

```text
MAE
RMSE
R²
Recall
F1-score

In [14]:
# RandomForestRegressor 하이퍼파라미터 튜닝 후보
# 140만 건 데이터라 전체 GridSearch는 오래 걸리므로,
# ParameterSampler로 일부 조합만 랜덤하게 실험한다.
param_dist = {
    "n_estimators": [50, 100],
    "max_depth": [14, 16, 18, 22],
    "min_samples_leaf": [10, 20, 50],
    "max_features": [0.5, "sqrt", 1.0],
}

N_TUNE_ITER = 8

sampler = list(
    ParameterSampler(
        param_dist,
        n_iter=N_TUNE_ITER,
        random_state=RANDOM_STATE,
    )
)

tuning_rows = []
best_params = None
best_mae = np.inf

for i, params in enumerate(sampler, start=1):
    print(f"[{i}/{N_TUNE_ITER}] params:", params)

    model = build_rf_pipeline(params)

    model.fit(train[FEATURES], train["target_min"])

    pred = model.predict(valid[FEATURES])

    metrics = regression_metrics(valid["target_min"], pred)

    tuning_rows.append({
        "trial": i,
        **params,
        **metrics,
    })

    if metrics["MAE"] < best_mae:
        best_mae = metrics["MAE"]
        best_params = params

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(tuning_results.round(4))

best_params

[1/8] params: {'n_estimators': 50, 'min_samples_leaf': 50, 'max_features': 0.5, 'max_depth': 14}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   10.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.2s finished


[2/8] params: {'n_estimators': 50, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 22}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    9.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.4s finished


[3/8] params: {'n_estimators': 50, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 16}


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    8.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.2s finished


[8/8] params: {'n_estimators': 50, 'min_samples_leaf': 50, 'max_features': 1.0, 'max_depth': 16}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   14.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   18.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.2s finished


,trial,n_estimators,min_samples_leaf,max_features,max_depth,MAE,Median_AE,RMSE,R2
0,6,50,20,1.0,18,16.5946,10.8776,25.6304,0.4970
1,3,50,10,0.5,16,16.8463,11.3011,26.0064,0.4821
2,2,50,20,sqrt,22,16.9708,11.3434,26.0312,0.4811
3,8,50,50,1.0,16,17.0406,11.2453,26.3461,0.4685
4,4,50,10,0.5,14,17.2291,11.6545,26.5943,0.4584
5,1,50,50,0.5,14,17.5085,11.7896,26.9994,0.4418
6,5,50,50,sqrt,16,17.5501,11.8421,27.0007,0.4417
7,7,50,50,sqrt,14,17.7524,12.0313,27.2885,0.4298


{'n_estimators': 50,
 'min_samples_leaf': 20,
 'max_features': 1.0,
 'max_depth': 18}

## 9. RandomForestRegressor 튜닝 결과 해석

RandomForestRegressor의 하이퍼파라미터를 `ParameterSampler` 기반 소규모 RandomizedSearch로 비교했다.  
튜닝 기준은 validation set의 MAE로 설정했다.

비교 결과, 가장 좋은 성능을 보인 조합은 다음과 같다.

```text
n_estimators = 50
max_depth = 18
min_samples_leaf = 20
max_features = 1.0

해당 조합의 validation 성능
MAE = 16.59분
RMSE = 25.63분
R² = 0.50

In [15]:
# RandomForestRegressor 튜닝 결과에서 선택한 최적 파라미터
best_params = {
    "n_estimators": 50,
    "min_samples_leaf": 20,
    "max_features": 1.0,
    "max_depth": 18,
}

# train과 validation을 합쳐 최종 학습 데이터로 사용
train_valid = pd.concat([train, valid], ignore_index=True)

# 최적 파라미터로 최종 RandomForest 모델 학습
final_rf_model = build_rf_pipeline(best_params)

final_rf_model.fit(
    train_valid[FEATURES],
    train_valid["target_min"],
)

# test set 예측
test_pred_rf_final = final_rf_model.predict(test[FEATURES])

# test set 최종 성능 평가
final_rf_test_metrics = evaluate_predictions(
    test,
    test_pred_rf_final,
    long_wait_threshold,
)

final_rf_test_metrics

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   25.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.3s finished


{'MAE': 16.483884074729804,
 'Median_AE': 10.787399649014347,
 'RMSE': np.float64(25.48641061442379),
 'R2': 0.49879260617146115,
 'risk_threshold': np.float64(94.14695333333344),
 'risk_Accuracy': 0.9249152912367954,
 'risk_Precision': 0.7742585338556239,
 'risk_Recall': 0.3338319741350191,
 'risk_F1': 0.4665183087194012,
 'risk_Confusion_Matrix': [[187982, 2017], [13805, 6918]]}

| 구분 | 건수 |
|---|---:|
| 정상 대기를 정상으로 예측 | 187,982 |
| 정상 대기를 장시간으로 잘못 경고 | 2,017 |
| 장시간 대기를 놓침 | 13,805 |
| 장시간 대기를 잡음 | 6,918 |

```text
정상 대기는 매우 잘 구분함
장시간 대기 경고는 정확한 편
하지만 실제 장시간 대기 중 놓치는 건이 많음

## 10. 최종 모델 학습 및 평가

튜닝에서 선택된 파라미터로 train+validation을 합쳐 최종 학습한 뒤, test set에서 최종 성능을 확인한다.

In [16]:
# train과 validation을 합쳐 최종 학습 데이터로 사용
train_valid = pd.concat([train, valid], ignore_index=True)

# RandomForest 튜닝에서 선택한 최적 파라미터
best_params = {
    "n_estimators": 50,
    "min_samples_leaf": 20,
    "max_features": 1.0,
    "max_depth": 18,
}

# 최종 RandomForest 모델 학습
final_model = build_rf_pipeline(best_params)

final_model.fit(
    train_valid[FEATURES],
    train_valid["target_min"],
)

# test set 예측
test_pred = final_model.predict(test[FEATURES])

# test set 최종 성능 평가
final_test_metrics = evaluate_predictions(
    test,
    test_pred,
    long_wait_threshold,
)

final_test_metrics

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   17.8s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   23.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.3s finished


{'MAE': 16.483884074729804,
 'Median_AE': 10.787399649014343,
 'RMSE': np.float64(25.48641061442379),
 'R2': 0.49879260617146115,
 'risk_threshold': np.float64(94.14695333333344),
 'risk_Accuracy': 0.9249152912367954,
 'risk_Precision': 0.7742585338556239,
 'risk_Recall': 0.3338319741350191,
 'risk_F1': 0.4665183087194012,
 'risk_Confusion_Matrix': [[187982, 2017], [13805, 6918]]}

최종 RandomForestRegressor 모델은 train과 validation set을 합친 데이터로 학습한 뒤, 독립적인 test set에서 평가하였다. test set 평가 결과 MAE는 16.48분, RMSE는 25.49분, R²는 0.499로 나타났다. 이는 조건별 중앙값을 사용하는 Baseline 2의 test MAE 17.51분, RMSE 29.20분, R² 0.342보다 개선된 결과다. 특히 RMSE가 감소한 점은 RandomForest 모델이 장시간 대기처럼 큰 오차가 발생하기 쉬운 구간을 baseline보다 더 잘 완화했음을 의미한다.

장시간 대기 기준은 train set의 접수→승차 90분위수인 94.15분으로 설정하였다. 이 기준에서 Precision은 0.774, Recall은 0.334, F1-score는 0.467이었다. 모델이 장시간 대기라고 예측한 경우 실제 장시간 대기일 가능성은 비교적 높았지만, 실제 장시간 대기 사례 중 약 33.4%만 탐지하여 Recall은 낮은 편이었다. 따라서 현재 모델은 장시간 대기 위험을 보수적으로 경고하는 모델로 해석된다.

## 10. model_group별 성능 확인

통합 모델이므로 전체 성능만 보면 안 된다. `임차택시_바로콜`, `특장차_바로콜` 각각의 성능을 따로 확인한다.

In [17]:
group_metrics = evaluate_by_group(test, test_pred, threshold_by_group)
display(group_metrics.round(4))

,model_group,rows,MAE,Median_AE,RMSE,R2,risk_threshold,risk_Accuracy,risk_Precision,risk_Recall,risk_F1
0,임차택시_바로콜,45860,15.8093,10.1104,25.3921,0.5534,87.2895,0.9334,0.8166,0.4289,0.5624
1,특장차_바로콜,164862,16.6715,10.9793,25.5126,0.4802,95.2808,0.9222,0.7560,0.3043,0.4340


그룹별 성능을 비교한 결과, 임차택시 바로콜의 예측 성능이 특장차 바로콜보다 전반적으로 높게 나타났다. 임차택시 바로콜의 test MAE는 15.81분, R²는 0.553인 반면, 특장차 바로콜의 MAE는 16.67분, R²는 0.480이었다. 이는 통합 모델 안에서도 임차택시 대기시간 패턴이 상대적으로 더 안정적으로 예측되고 있음을 의미한다.

장시간 대기 탐지 성능에서도 임차택시가 더 높았다. 임차택시 바로콜의 Recall은 0.429, F1-score는 0.562였으나, 특장차 바로콜의 Recall은 0.304, F1-score는 0.434로 낮았다. 특히 특장차는 장시간 대기 기준이 약 95.28분으로 임차택시의 87.29분보다 높았음에도 실제 장시간 대기를 탐지하는 비율이 낮았다.

따라서 통합 모델은 전체적인 예상 대기시간 산출에는 사용할 수 있으나, 특장차 바로콜의 장시간 대기 위험 안내는 보수적으로 해석해야 한다. 향후 특장차 모델 성능 개선을 위해서는 실시간 대기열, 배차 가능 차량 수, 휠체어 가점, 병원·복지시설 목적 여부 등 추가 변수를 검토할 필요가 있다.

## 11. 결과 저장

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, MODEL_DIR / "baro_wait_time_hgb_model.joblib")

metadata = {
    "model_name": "baro_wait_time_hgb_model",
    "target": "접수→승차_분",
    "features": FEATURES,
    "best_params": best_params,
    "target_cap": None,
    "overall_long_wait_threshold_train_p90": float(long_wait_threshold),
    "group_long_wait_threshold_train_p90": {k: float(v) for k, v in threshold_by_group.items()},
    "extreme_wait_threshold": 180.0,
}

with open(MODEL_DIR / "baro_wait_time_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

pd.DataFrame([final_test_metrics]).to_csv(
    REPORT_DIR / "notebook_final_test_metrics.csv", index=False, encoding="utf-8-sig"
)
group_metrics.to_csv(
    REPORT_DIR / "notebook_final_test_metrics_by_group.csv", index=False, encoding="utf-8-sig"
)
tuning_results.to_csv(
    REPORT_DIR / "notebook_tuning_results.csv", index=False, encoding="utf-8-sig"
)

MODEL_DIR, REPORT_DIR

## 12. 예측 함수 예시

In [ ]:
def make_features_for_prediction(model_group, request_datetime, origin_gu, destination_gu):
    dt = pd.to_datetime(request_datetime)
    month = dt.month
    hour = dt.hour
    dayofweek = dt.dayofweek
    return pd.DataFrame([{
        "model_group": model_group,
        "출발구": origin_gu,
        "목적구": destination_gu,
        "hour": hour,
        "dayofweek": dayofweek,
        "month": month,
        "is_weekend": int(dayofweek in [5, 6]),
        "is_night": int(0 <= hour <= 6),
        "is_commute": int((7 <= hour <= 9) or (17 <= hour <= 19)),
    }])

def predict_wait_time(model_group, request_datetime, origin_gu, destination_gu):
    X = make_features_for_prediction(model_group, request_datetime, origin_gu, destination_gu)
    pred = float(final_model.predict(X[FEATURES])[0])
    threshold = threshold_by_group.get(model_group, long_wait_threshold)
    if pred >= threshold:
        risk = "높음"
    elif pred >= threshold * 0.75:
        risk = "보통"
    else:
        risk = "낮음"
    return {
        "model_group": model_group,
        "origin_gu": origin_gu,
        "destination_gu": destination_gu,
        "expected_wait_min": round(pred, 1),
        "risk_level": risk,
        "target_definition": "접수→승차 예상 대기시간",
        "vehicle_travel_time_source": "지도/경로 API",
    }

predict_wait_time("특장차_바로콜", "2026-09-10 05:00", "관악구", "서초구")

## 13. 샘플 예측 결과 확인

In [ ]:
sample = test.sample(n=20, random_state=RANDOM_STATE).copy()
sample["prediction_min"] = final_model.predict(sample[FEATURES])

display(sample[[
    "model_group", "접수일시", "출발구", "출발동", "목적구", "목적동",
    "target_min", "prediction_min", "is_extreme_wait", "접수_배차_분", "배차_승차_분"
]].round(2))